# Evaluate sleep fragmentation

### Import recordings

Load packages

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os
from IPython.display import display
from ipyfilechooser import FileChooser
from scipy.stats import zscore
import json
import matplotlib.cm as cm
import IPython
import ast
from scipy.interpolate import interp1d
import pickle
from datetime import datetime

Load functions

In [ ]:
def flip_headstage(LFPs_df, ID):
    rec_ch_list_ID = LFPs_df.columns-ID*32
    rec_ch_list_mouse = [value for value in rec_ch_list_ID if 0 <= value <= 31]
    i = np.argmax(rec_ch_list_ID>=0)
    inverted_chs = np.concatenate([range(16,32,1), range(0,16,1)], axis=0)
    LFPs_df_mouse=LFPs_df.iloc[:,i:i+len(rec_ch_list_mouse)]
    flipped_ch=(inverted_chs[LFPs_df_mouse.columns-(ID*32)])+(ID*32)
    LFPs_df.columns.values[i:i+len(rec_ch_list_mouse)] = flipped_ch
    LFPs_df = LFPs_df.sort_index(axis=1)
    return LFPs_df


Process

In [ ]:
folder_path = "//10.69.168.1/crnldata/forgetting/Carla/OpenEphysData/Sleep/4 mois/TraitViolet_TraitRouge_TraitVert_2025-02-28_17-57-01_Sleep-4mois/"

Summary_table=pd.DataFrame()
counter=0

#Load LFP coordinates 
notebook_path = Path("/".join(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[-5:]))
Channels = f'{notebook_path.parent}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0)
all_LFPcoordinates = pd.read_csv('_LFP_coordinates_of_all_mice.csv', sep=';', index_col=0)

#print("Transform .dat to .pkl if needed...")
for root, dirs, files in os.walk(folder_path):
    datfiles = [f for f in files if f.endswith(('.dat'))]
    for datfile in datfiles:
        #Path(root).parents[6].name
        datfilepath = os.path.join(root, datfile)
        datfilepath = Path(os.path.normpath(datfilepath))
        # Load LFP file 
        DataRec = np.fromfile(datfilepath, dtype="int16")
        print(f'.dat file found = {datfilepath}')
        Metadatafilepath = Path(os.path.join(datfilepath.parent.parent.parent, f'structure.oebin'))
        with open(Metadatafilepath) as f:
            metadata = json.load(f)
        samplerate = metadata['continuous'][0]['sample_rate']  
        numchannel = metadata['continuous'][0]['num_channels'] 
        rec_ch_list_origin = np.array([int(''.join(c for c in metadata['continuous'][0]['channels'][x]['channel_name'] if c.isdigit()))-1 for x in range(0, len(metadata['continuous'][0]['channels']))])
        rec_ch_list = np.array([int(''.join(filter(str.isdigit, c['channel_name']))) - 1  for c in metadata['continuous'][0]['channels'] if c['channel_name'].startswith('CH')])
        DataRec = DataRec.reshape(-1,numchannel)
        numchannel = len(rec_ch_list)
        DataRec = DataRec[:,:numchannel]

        # Load LFPs timestamps 
        for file_pathTS in datfilepath.parent.parent.parent.glob('**/continuous/*/timeStamps.npy'):
            print('LFPs timestamps file found')
            LFPtimestamps = np.load(file_pathTS) 
        LFPs_df=pd.DataFrame(DataRec, columns=rec_ch_list) 

        # Identify mouse
        mice = []
        pos_mice = []
        for mice_name in all_LFPcoordinates.index:
            if mice_name in datfilepath.__str__():
                mice.append(mice_name)
                pos_mice.append(datfilepath.__str__().find(mice_name)) 
        mice = [x for _, x in sorted(zip(pos_mice, mice))] # sort mouse in the same order as they appear in the path

        print(mice)
        
        for ID, mouse in enumerate(mice):
            
            mouse_age = datfilepath.parents[6].name.replace(' ', '')

            if mouse == 'TraitViolet' and mouse_age =='4mois':
                print('Flipping the headstage !!!! ')
                LFPs_df = flip_headstage(LFPs_df=LFPs_df, ID=ID)
                rec_ch_list =LFPs_df.keys().tolist()
                
            all_LFPcoordinates = all_LFPcoordinates.astype(str)
            for region in all_LFPcoordinates.loc[mouse].index:
                locals()[region] = []
                locals()[f'{region}_ch'] = []

            RecordedArea = []
            ChoosenChannels = []
            combined = []

            rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
            for rec_ch in rec_ch_list_mouse:
                for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
                    region = all_LFPcoordinates.loc[mouse].index[idx]
                    if LFPcoord_str != 'nan':
                        LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region                 
                        num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
                        if len(num_ch) > 0:
                            region = all_LFPcoordinates.loc[mouse].index[idx]
                            LFP = locals()[region]
                            LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                            locals()[region] = LFP
                            locals()[f'{region}_ch'].append(rec_ch)
                            break
                        continue

            for region in all_LFPcoordinates.loc[mouse].index:
                LFP = locals()[region]
                LFP_ch = locals()[f'{region}_ch']
                if len(LFP) > 0:
                    combined = zscore(LFP[:,np.newaxis]) if len(combined) == 0 else np.append(combined, zscore(LFP[:,np.newaxis]), axis=1)
                    RecordedArea.append(region) 
                    ChoosenChannels.append(LFP_ch) 

            print(mouse)
            print(RecordedArea)
            print(ChoosenChannels) 


            ###### Save .pkl files ######

            LFPs_df_mouse=LFPs_df[rec_ch_list_mouse]
            LFPs_df_mouse = LFPs_df_mouse.copy()
            LFPs_df_mouse.rename(columns={x: x - (ID * 32) for x in rec_ch_list_mouse}, inplace=True)
            #LFPs_df.to_pickle(f'{LFPfile.parent}/DataFrame_rawdataDS.pkl')
            directory = f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/{mouse}/{mouse_age}/'
            file_name = 'DataFrame_rawdataDS.pkl'
            file_path = os.path.join(directory, file_name)
            os.makedirs(directory, exist_ok=True)    
            LFPs_df_mouse.to_pickle(file_path)


            ###### Define Wake episode with EMG ######
            if len(EMG) == 0:
                if not len(S1) == 0:
                    print('EMG from S1')
                    EMG = S1
                elif not len(oPFC) == 0:
                    print('EMG from oPFC')
                    EMG = oPFC
                elif not len(PFC) == 0:
                    print('EMG from PFC')
                    EMG = PFC

            # Filter parameter :
            f_lowcut = 200.
            f_hicut = 400.
            N = 4
            fs = 1000
            nyq = 0.5 * fs
            Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction

            # Filter creation :
            b, a = signal.butter(N, Wn, 'band')
            filt_EMG = signal.filtfilt(b, a, EMG)

            # Parameter and computation of CWT
            w = 4.
            freq = np.linspace(200, 400, 50)
            widths = w*fs / (2*freq*np.pi)
            EMGcwt = signal.cwt(EMG, signal.morlet2, widths, w=w)

            # Projection calculation
            absEMGcwt = np.absolute(EMGcwt)
            proj_EMGcwt = np.sum(absEMGcwt, axis = 0)/50
            mproj_EMGcwt = np.mean(proj_EMGcwt)
            sdproj_EMGcwt = np.std(proj_EMGcwt)

            LowFactorSd=1
            HighFactorSd=3

            # Assigning values wake (1, 2) and sleep (0)
            numpnts = EMG.size
            EMGstatusRaw = np.zeros(numpnts)
            for ind in range(numpnts):
                if proj_EMGcwt[ind]<(mproj_EMGcwt + LowFactorSd*sdproj_EMGcwt): 
                    EMGstatusRaw[ind] = 0
                elif proj_EMGcwt[ind]>(mproj_EMGcwt + HighFactorSd*sdproj_EMGcwt):
                    EMGstatusRaw[ind] = 2
                else:
                    EMGstatusRaw[ind] = 1

            # Expanding borders for wake (1, 2) and sleep (0) to ±1 s around detected muscular activity
            EMGstatusRaw2 = np.zeros(numpnts)
            for ind in range(numpnts):
                if EMGstatusRaw[ind]>1:
                    EMGstatusRaw2[ind-1000:ind+1000] = 2
                elif EMGstatusRaw[ind]==1:
                    for ind2 in range(ind-1000, ind+1000):
                        if ind2==numpnts:
                            break
                        elif EMGstatusRaw2[ind2]<2:
                            EMGstatusRaw2[ind2] = 1

            EMGStatusBoolLib = (EMGstatusRaw2>1)
            EMGStatusBoolCons = (EMGstatusRaw2>0)

            EMG_scoring = pd.DataFrame({
                "time": range(len(EMGStatusBoolCons)),
                "duration": 1,
                "label": ["Wake" if x else "Sleep" for x in EMGStatusBoolCons]
            })

            EMG_scoring_filename = os.path.join(Path(file_path).parent, f'EMG_scoring.csv')
            EMG_scoring.to_csv(EMG_scoring_filename, index=False)

In [ ]:
LFPs_df=pd.DataFrame(DataRec, columns=rec_ch_list) 
for ID, mouse in enumerate(mice):
            
    mouse_age = datfilepath.parents[6].name.replace(' ', '')

    if mouse == 'TraitViolet' and mouse_age =='4mois':
        print('Flipping the headstage !!!! ')
        LFPs_df = flip_headstage(LFPs_df=LFPs_df, ID=ID)
        rec_ch_list =LFPs_df.keys().tolist()
        
    all_LFPcoordinates = all_LFPcoordinates.astype(str)
    for region in all_LFPcoordinates.loc[mouse].index:
        locals()[region] = []
        locals()[f'{region}_ch'] = []

    RecordedArea = []
    ChoosenChannels = []
    combined = []

    rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
    for rec_ch in rec_ch_list_mouse:
        for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
            region = all_LFPcoordinates.loc[mouse].index[idx]
            if LFPcoord_str != 'nan':
                LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region                 
                num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
                if len(num_ch) > 0:
                    region = all_LFPcoordinates.loc[mouse].index[idx]
                    LFP = locals()[region]
                    LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                    locals()[region] = LFP
                    locals()[f'{region}_ch'].append(rec_ch)
                    break
                continue

    for region in all_LFPcoordinates.loc[mouse].index:
        LFP = locals()[region]
        LFP_ch = locals()[f'{region}_ch']
        if len(LFP) > 0:
            combined = zscore(LFP[:,np.newaxis]) if len(combined) == 0 else np.append(combined, zscore(LFP[:,np.newaxis]), axis=1)
            RecordedArea.append(region) 
            ChoosenChannels.append(LFP_ch) 

    print(mouse)
    print(RecordedArea)
    print(ChoosenChannels) 


    ###### Save .pkl files ######

    LFPs_df_mouse=LFPs_df[rec_ch_list_mouse]
    LFPs_df_mouse = LFPs_df_mouse.copy()
    LFPs_df_mouse.rename(columns={x: x - (ID * 32) for x in rec_ch_list_mouse}, inplace=True)
    #LFPs_df.to_pickle(f'{LFPfile.parent}/DataFrame_rawdataDS.pkl')
    directory = f'//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings/{mouse}/{mouse_age}/'
    file_name = 'DataFrame_rawdataDS.pkl'
    file_path = os.path.join(directory, file_name)
    os.makedirs(directory, exist_ok=True)    
    LFPs_df_mouse.to_pickle(file_path)


    ###### Define Wake episode with EMG ######
    if len(EMG) == 0:
        if not len(S1) == 0:
            print('EMG from S1')
            EMG = S1
        elif not len(oPFC) == 0:
            print('EMG from oPFC')
            EMG = oPFC
        elif not len(PFC) == 0:
            print('EMG from PFC')
            EMG = PFC

    # Filter parameter :
    f_lowcut = 200.
    f_hicut = 400.
    N = 4
    fs = 1000
    nyq = 0.5 * fs
    Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction

    # Filter creation :
    b, a = signal.butter(N, Wn, 'band')
    filt_EMG = signal.filtfilt(b, a, EMG)

    # Parameter and computation of CWT
    w = 4.
    freq = np.linspace(200, 400, 50)
    widths = w*fs / (2*freq*np.pi)
    EMGcwt = signal.cwt(EMG, signal.morlet2, widths, w=w)

    # Projection calculation
    absEMGcwt = np.absolute(EMGcwt)
    proj_EMGcwt = np.sum(absEMGcwt, axis = 0)/50
    mproj_EMGcwt = np.mean(proj_EMGcwt)
    sdproj_EMGcwt = np.std(proj_EMGcwt)

    LowFactorSd=1
    HighFactorSd=3

    # Assigning values wake (1, 2) and sleep (0)
    numpnts = EMG.size
    EMGstatusRaw = np.zeros(numpnts)
    for ind in range(numpnts):
        if proj_EMGcwt[ind]<(mproj_EMGcwt + LowFactorSd*sdproj_EMGcwt): 
            EMGstatusRaw[ind] = 0
        elif proj_EMGcwt[ind]>(mproj_EMGcwt + HighFactorSd*sdproj_EMGcwt):
            EMGstatusRaw[ind] = 2
        else:
            EMGstatusRaw[ind] = 1

    # Expanding borders for wake (1, 2) and sleep (0) to ±1 s around detected muscular activity
    EMGstatusRaw2 = np.zeros(numpnts)
    for ind in range(numpnts):
        if EMGstatusRaw[ind]>1:
            EMGstatusRaw2[ind-1000:ind+1000] = 2
        elif EMGstatusRaw[ind]==1:
            for ind2 in range(ind-1000, ind+1000):
                if ind2==numpnts:
                    break
                elif EMGstatusRaw2[ind2]<2:
                    EMGstatusRaw2[ind2] = 1

    EMGStatusBoolLib = (EMGstatusRaw2>1)
    EMGStatusBoolCons = (EMGstatusRaw2>0)

    EMG_scoring = pd.DataFrame({
        "time": range(len(EMGStatusBoolCons)),
        "duration": 1,
        "label": ["Wake" if x else "Sleep" for x in EMGStatusBoolCons]
    })

    EMG_scoring_filename = os.path.join(Path(file_path).parent, f'EMG_scoring.csv')
    EMG_scoring.to_csv(EMG_scoring_filename, index=False)

In [ ]:

            

            # Store results in table
            Summary_table.loc[counter, 'Mouse_ID'] = mouse
            Summary_table.loc[counter, 'Genotype'] = LFPfilepath.parents[8].name
            Summary_table.loc[counter, 'Age'] = LFPfilepath.parents[5].name.split('_')[-1]
            Summary_table.loc[counter, 'Session'] = LFPfilepath.parents[5].name.split('_')[1]
            Summary_table.loc[counter, 'Session_time'] = LFPfilepath.parents[5].name.split('_')[2]
            Summary_table.loc[counter, 'ChR2inj_site'] = LFPfilepath.parents[9].name
            Summary_table.loc[counter, 'Sex'] = LFPfilepath.parents[7].name
            Summary_table.loc[counter, 'Region'] = region_name.split('_')[0]
            Summary_table.loc[counter, 'Region_ch'] = region_name
            Summary_table.loc[counter, 'Stim_dur'] = stim_duration
            Summary_table.loc[counter, 'Nb_stims'] = nb_stims
            Summary_table.loc[counter, 'Amplitude_at_t0'] = result['Amplitude_at_t0']
            Summary_table.loc[counter, 'Amplitude_max'] = result['Amplitude_max']
            Summary_table.loc[counter, 'Time_Amp_max'] = result['Time_Amp_max']
            Summary_table.loc[counter, 'Amplitude_min'] = result['Amplitude_min']
            Summary_table.loc[counter, 'Time_Amp_min'] = result['Time_Amp_min']
            Summary_table.loc[counter, 'Negative_slope'] = result['Negative_slope']

            if not region_name in MeanSignal_dict[mouse]:
                MeanSignal_dict[mouse][region_name]={}

            MeanSignal_dict[mouse][region_name][LFPfilepath.parents[5].name.split('_')[-1]]= result['mean_signal']
            
            counter+=1

filenameOut = f'{folder_path}/Summary_table.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    Summary_table.to_excel(writer, index=False)

with open(f'{folder_path}/MeanSignal_dict.pkl', 'wb') as f:
    pickle.dump(MeanSignal_dict, f)

print(f"Analyse terminée ! Tableau sauvegardé dans : {filenameOut}")
print(f"Nombre total d'essais analysés : {counter}") 